# Stage 6 — Base ASAG Semantic & Lexical Relationships

**Project:** Automatic Short Answer Grading using Semantic and Dependency Analysis  
**Repository:** `proposition-level-asag`  
**Current Stage:** Stage 6 — Semantic / Lexical Relationships  

---

### Purpose
This notebook demonstrates the **Stage 6 Component** for the Base ASAG system.

In previous stages, we built:
- **Stage 3:** Linguistic preprocessing (tokenization, lemmatization, POS tagging).
- **Stage 4:** Dependency parsing (syntactic heads, dependency labels).
- **Stage 5:** Dependency structures (raw arcs and canonical `(subject, predicate, object)` triples).

**Stage 6 introduces term-to-term semantic and lexical relationship analysis.**  
When comparing student answers against reference answers, exact string matches are insufficient because students use different inflectional forms, synonyms, or related vocabulary.

Stage 6 implements a **4-level hierarchical comparator**:
1. **Level 1 — Exact Match:** Surface-form identity (`record` $\leftrightarrow$ `record`).
2. **Level 2 — Lemma Match:** Canonical base-form identity (`identifies` $\leftrightarrow$ `identified` $\rightarrow$ `identify`).
3. **Level 3 — POS Compatibility Guard:** Rejects cross-part-of-speech comparisons before querying taxonomic networks (e.g. `identifies` [VERB] vs `table` [NOUN]).
4. **Level 4 — WordNet Semantic Similarity:** For POS-compatible noun, verb, adjective, or adverb pairs, looks up synsets in NLTK WordNet and calculates maximum path similarity.

> **CRITICAL RESEARCH BOUNDARIES:**  
> - Stage 6 operates strictly at the **term/token level** (word-to-word).
> - It does **not** perform triple matching (reserved for Stage 7).
> - It does **not** compute sentence-level cosine similarity or sentence embeddings.
> - It does **not** calculate student grades or evaluate textual entailment.

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> Synthetic demonstration examples for pipeline verification only.

## 1. Imports and Setup

We import pandas for reporting and the Stage 6 public API:
- `LexicalSemanticComparator`
- `SemanticRelation`
- `map_pos_to_wordnet`
- `is_pos_compatible`

We also import Stage 3 Preprocessing, Stage 4 Dependency Parsing, and Stage 5 Triple Extraction to verify seamless integration across pipeline stages.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure repository root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from base_asag.preprocessing import TextPreprocessor
from base_asag.dependency import DependencyParser, DependencyTripleExtractor
from base_asag.semantic import (
    LexicalSemanticComparator,
    SemanticRelation,
    map_pos_to_wordnet,
    is_pos_compatible,
)

# Initialize the comparator
comparator = LexicalSemanticComparator()

print("Initialized LexicalSemanticComparator successfully.")
print(f"WordNet corpus reader: {comparator.wn}")

Initialized LexicalSemanticComparator successfully.
WordNet corpus reader: <WordNetCorpusReader in '/Users/sarang/nltk_data/corpora/wordnet.zip/wordnet/'>


## 2. Hierarchy Level 1: Exact Surface Match

The fastest, most reliable match is surface-form identity.
- When two words have identical text (case-insensitive), `similarity_score = 1.0` and `method = 'exact_match'`.
- WordNet lookup is **completely bypassed**, saving compute and avoiding semantic drift.

In [2]:
rel_exact = comparator.compare("record", "record", source_pos="NOUN", target_pos="NOUN")
print("Comparison: 'record' vs 'record'")
print(f"  Method:            {rel_exact.method}")
print(f"  Exact Match:       {rel_exact.exact_match}")
print(f"  Lemma Match:       {rel_exact.lemma_match}")
print(f"  Similarity Score:  {rel_exact.similarity_score}")
print(f"  POS Compatible:    {rel_exact.pos_compatible}")
print(f"  Semantic Relation: {rel_exact.semantic_relation}")

Comparison: 'record' vs 'record'
  Method:            exact_match
  Exact Match:       True
  Lemma Match:       True
  Similarity Score:  1.0
  POS Compatible:    True
  Semantic Relation: EXACT_MATCH


## 3. Hierarchy Level 2: Lemma Match

When surface strings differ due to grammatical inflection (e.g., tense, pluralization), the comparator checks whether their lemmas are identical.
- For example, `identifies` (third-person singular present) and `identified` (past tense) both share the lemma `identify`.
- Similarly, `keys` (plural noun) and `key` (singular noun) both share the lemma `key`.
- Lemma matching resolves inflectional variation deterministically with `similarity_score = 1.0` and `method = 'lemma_match'`, without querying WordNet.

In [3]:
rel_lemma1 = comparator.compare(
    "identifies", "identified",
    source_lemma="identify", target_lemma="identify",
    source_pos="VERB", target_pos="VERB"
)

rel_lemma2 = comparator.compare(
    "keys", "key",
    source_lemma="key", target_lemma="key",
    source_pos="NOUN", target_pos="NOUN"
)

df_lemmas = pd.DataFrame([rel_lemma1.to_dict(), rel_lemma2.to_dict()])
df_lemmas[["source_text", "target_text", "source_lemma", "target_lemma", "method", "similarity_score", "semantic_relation"]]

,source_text,target_text,source_lemma,target_lemma,method,similarity_score,semantic_relation
0,identifies,identified,identify,identify,lemma_match,1.0,LEMMA_MATCH
1,keys,key,key,key,lemma_match,1.0,LEMMA_MATCH


## 4. Hierarchy Level 3: POS Compatibility Guard

In dependency-based ASAG, comparing words of different grammatical categories (e.g., a verb against a noun) is linguistically invalid and introduces spurious alignments.

Before querying WordNet:
- Both Universal POS tags (`NOUN`, `VERB`, `ADJ`, `ADV`, `PROPN`) and Penn Treebank tags are mapped to WordNet synset categories (`n`, `v`, `a`, `r`).
- If one term is a `VERB` and the other is a `NOUN`, `pos_compatible = False`.
- The comparison is rejected early with `similarity_score = 0.0` and `method = 'pos_incompatible'`.

In [4]:
rel_incomp1 = comparator.compare(
    "identifies", "table",
    source_pos="VERB", target_pos="NOUN"
)

rel_incomp2 = comparator.compare(
    "uniquely", "unique",
    source_pos="ADV", target_pos="ADJ"
)

df_incomp = pd.DataFrame([rel_incomp1.to_dict(), rel_incomp2.to_dict()])
df_incomp[["source_text", "target_text", "source_pos", "target_pos", "pos_compatible", "method", "similarity_score", "semantic_relation"]]

,source_text,target_text,source_pos,target_pos,pos_compatible,method,similarity_score,semantic_relation
0,identifies,table,VERB,NOUN,False,pos_incompatible,0.0,POS_INCOMPATIBLE
1,uniquely,unique,ADV,ADJ,False,pos_incompatible,0.0,POS_INCOMPATIBLE


## 5. Hierarchy Level 4: WordNet True Synonym

When surface forms and lemmas differ, but the terms share a common grammatical category, the comparator queries WordNet.

If two terms share an identical synset, they are synonyms:
- Example: `car` and `automobile` (both `NOUN`).
- Result: `similarity_score = 1.0`, `method = 'wordnet_synonym'`.
- Synsets: `car.n.01`.

In [5]:
rel_syn = comparator.compare("car", "automobile", source_pos="NOUN", target_pos="NOUN")
print("Comparison: 'car' vs 'automobile'")
print(f"  Method:            {rel_syn.method}")
print(f"  Semantic Relation: {rel_syn.semantic_relation}")
print(f"  Similarity Score:  {rel_syn.similarity_score}")
print(f"  Synset Source:     {rel_syn.synset_source}")
print(f"  Synset Target:     {rel_syn.synset_target}")

Comparison: 'car' vs 'automobile'
  Method:            wordnet_synonym
  Semantic Relation: WORDNET_SYNONYM
  Similarity Score:  1.0
  Synset Source:     car.n.01
  Synset Target:     car.n.01


## 6. WordNet Lookup for Domain Terms: `record` vs `row`

In relational database terminology:
- *Reference Answer:* "A primary key uniquely identifies each **record** in a table."
- *Student Answer:* "A primary key uniquely identifies every **row** in a table."

To a human grader, `record` and `row` are interchangeable synonyms in this domain.  
Let us inspect what general-purpose WordNet actually returns:

In [6]:
rel_rec_row = comparator.compare("record", "row", source_pos="NOUN", target_pos="NOUN")

print("WordNet Evaluation of 'record' vs 'row':")
print(f"  Method:            {rel_rec_row.method}")
print(f"  Semantic Relation: {rel_rec_row.semantic_relation}")
print(f"  Similarity Score:  {rel_rec_row.similarity_score}")
print(f"  Synset Source:     {rel_rec_row.synset_source}")
print(f"  Synset Target:     {rel_rec_row.synset_target}")

WordNet Evaluation of 'record' vs 'row':
  Method:            wordnet_path
  Semantic Relation: WORDNET_PATH
  Similarity Score:  0.125
  Synset Source:     record.n.04
  Synset Target:     quarrel.n.01


### Critical Observation: The General Ontology Gap

> **RESEARCH FINDING:**  
> WordNet assigns `record` and `row` a path similarity of only **0.125**, with method `wordnet_path`.
>
> **Why does this occur?**  
> WordNet is a general-purpose lexical database created for broad English, not computer science or database theory:
> - In WordNet, `row` denotes a physical linear arrangement (`row.n.01`) or a dispute/quarrel (`quarrel.n.01`).
> - `record` denotes an audio phonograph disc (`phonograph_record.n.01`) or a compilation of facts (`record.n.01`).
> - There is **no synset in general WordNet that equates a database record to a table row**.
>
> In accordance with our **Research Integrity Rules**, we do **not** hardcode a fake synonym rule or claim that WordNet solves this. We report the exact output of the algorithm and document this domain limitation.

## 7. WordNet Lookup for Related Domain Terms: `record` vs `table`

We compare two terms from the same domain that are related concepts, but not synonyms: `record` and `table`.

In [7]:
rel_rec_tbl = comparator.compare("record", "table", source_pos="NOUN", target_pos="NOUN")

print("WordNet Evaluation of 'record' vs 'table':")
print(f"  Method:            {rel_rec_tbl.method}")
print(f"  Semantic Relation: {rel_rec_tbl.semantic_relation}")
print(f"  Similarity Score:  {rel_rec_tbl.similarity_score}")
print(f"  Synset Source:     {rel_rec_tbl.synset_source}")
print(f"  Synset Target:     {rel_rec_tbl.synset_target}")

WordNet Evaluation of 'record' vs 'table':
  Method:            wordnet_path
  Semantic Relation: WORDNET_PATH
  Similarity Score:  0.125
  Synset Source:     record.n.07
  Synset Target:     table.n.01


## 8. WordNet Lookup for Completely Unrelated Term: `record` vs `photosynthesis`

To verify that the comparator does not assign spurious similarity to distant concepts, we compare `record` (NOUN) against `photosynthesis` (NOUN).

In [8]:
rel_rec_photo = comparator.compare("record", "photosynthesis", source_pos="NOUN", target_pos="NOUN")

print("WordNet Evaluation of 'record' vs 'photosynthesis':")
print(f"  Method:            {rel_rec_photo.method}")
print(f"  Semantic Relation: {rel_rec_photo.semantic_relation}")
print(f"  Similarity Score:  {rel_rec_photo.similarity_score}")
print(f"  Synset Source:     {rel_rec_photo.synset_source}")
print(f"  Synset Target:     {rel_rec_photo.synset_target}")

WordNet Evaluation of 'record' vs 'photosynthesis':
  Method:            wordnet_path
  Semantic Relation: WORDNET_PATH
  Similarity Score:  0.1
  Synset Source:     record.n.07
  Synset Target:     photosynthesis.n.01


## 9. Direct Integration with Stage 5 Triples & Parsed Tokens

In Stage 5, the dependency parser and triple extractor produced structured `CanonicalRelation` objects containing `(subject, predicate, object)`.

Here, we demonstrate that `LexicalSemanticComparator` directly accepts tokens from Stage 4/5 (`ParsedToken`, `TokenInfo`) and compares the constituent slots of reference and student triples.

In [9]:
# Extract triples for Reference and Student answers from Stage 5
parser = DependencyParser(model_name="en_core_web_sm")
extractor = DependencyTripleExtractor()

ref_sent = parser.parse_sentence("A primary key uniquely identifies each record in a table.")
stu_sent = parser.parse_sentence("A primary key uniquely identifies every row in a table.")

ref_triples = extractor.extract(ref_sent)
stu_triples = extractor.extract(stu_sent)

ref_spo = ref_triples.canonical_relations[0]
stu_spo = stu_triples.canonical_relations[0]

print(f"Reference Triple: ({ref_spo.subject}, {ref_spo.predicate}, {ref_spo.object})")
print(f"Student Triple:   ({stu_spo.subject}, {stu_spo.predicate}, {stu_spo.object})\n")

# Retrieve ParsedToken objects from Stage 4/5
ref_subj_tok = ref_sent.get_token(ref_spo.subject_index)
stu_subj_tok = stu_sent.get_token(stu_spo.subject_index)

ref_pred_tok = ref_sent.get_token(ref_spo.predicate_index)
stu_pred_tok = stu_sent.get_token(stu_spo.predicate_index)

ref_obj_tok = ref_sent.get_token(ref_spo.object_index)
stu_obj_tok = stu_sent.get_token(stu_spo.object_index)

# Compare corresponding slot tokens directly
slot_comparisons = [
    ("subject", comparator.compare(ref_subj_tok, stu_subj_tok)),
    ("predicate", comparator.compare(ref_pred_tok, stu_pred_tok)),
    ("object", comparator.compare(ref_obj_tok, stu_obj_tok)),
]

rows = []
for slot, rel in slot_comparisons:
    rows.append({
        "slot": slot,
        "ref_token": rel.source_text,
        "stu_token": rel.target_text,
        "ref_pos": rel.source_pos,
        "stu_pos": rel.target_pos,
        "method": rel.method,
        "similarity_score": rel.similarity_score,
        "semantic_relation": rel.semantic_relation,
    })

df_triple_comparison = pd.DataFrame(rows)
df_triple_comparison

Reference Triple: (key, identifies, record)
Student Triple:   (key, identifies, row)



,slot,ref_token,stu_token,ref_pos,stu_pos,method,similarity_score,semantic_relation
0,subject,key,key,NOUN,NOUN,exact_match,1.000,EXACT_MATCH
1,predicate,identifies,identifies,VERB,VERB,exact_match,1.000,EXACT_MATCH
2,object,record,row,NOUN,NOUN,wordnet_path,0.125,WORDNET_PATH


## 10. Summary Matrix of Demonstration Comparisons

We compile all demonstration pairs into a unified evaluation matrix to observe the behavior across all 4 hierarchy levels.

In [10]:
demo_pairs = [
    ("record", "record", "NOUN", "NOUN", "Level 1: Exact Surface Match"),
    ("identifies", "identified", "VERB", "VERB", "Level 2: Lemma Match"),
    ("keys", "key", "NOUN", "NOUN", "Level 2: Lemma Match"),
    ("identifies", "table", "VERB", "NOUN", "Level 3: POS Incompatible"),
    ("uniquely", "unique", "ADV", "ADJ", "Level 3: POS Incompatible"),
    ("car", "automobile", "NOUN", "NOUN", "Level 4: WordNet True Synonym"),
    ("record", "row", "NOUN", "NOUN", "Level 4: WordNet Domain Term Gap"),
    ("record", "table", "NOUN", "NOUN", "Level 4: WordNet Domain Related"),
    ("record", "photosynthesis", "NOUN", "NOUN", "Level 4: WordNet Distant/Unrelated"),
]

summary_rows = []
for src, tgt, s_pos, t_pos, desc in demo_pairs:
    res = comparator.compare(src, tgt, source_pos=s_pos, target_pos=t_pos)
    summary_rows.append({
        "Description": desc,
        "Source": src,
        "Target": tgt,
        "POS": f"{s_pos} - {t_pos}",
        "Method": res.method,
        "Score": round(res.similarity_score, 4),
        "Relation": res.semantic_relation,
    })

df_summary = pd.DataFrame(summary_rows)
df_summary

,Description,Source,Target,POS,Method,Score,Relation
0,Level 1: Exact Surface Match,record,record,NOUN - NOUN,exact_match,1.000,EXACT_MATCH
1,Level 2: Lemma Match,identifies,identified,VERB - VERB,wordnet_synonym,1.000,WORDNET_SYNONYM
2,Level 2: Lemma Match,keys,key,NOUN - NOUN,wordnet_synonym,1.000,WORDNET_SYNONYM
3,Level 3: POS Incompatible,identifies,table,VERB - NOUN,pos_incompatible,0.000,POS_INCOMPATIBLE
4,Level 3: POS Incompatible,uniquely,unique,ADV - ADJ,pos_incompatible,0.000,POS_INCOMPATIBLE
5,Level 4: WordNet True Synonym,car,automobile,NOUN - NOUN,wordnet_synonym,1.000,WORDNET_SYNONYM
6,Level 4: WordNet Domain Term Gap,record,row,NOUN - NOUN,wordnet_path,0.125,WORDNET_PATH
7,Level 4: WordNet Domain Related,record,table,NOUN - NOUN,wordnet_path,0.125,WORDNET_PATH
8,Level 4: WordNet Distant/Unrelated,record,photosynthesis,NOUN - NOUN,wordnet_path,0.100,WORDNET_PATH


## 11. Research Summary & Boundary Enforcement

### What Stage 6 Delivers
1. **Hierarchical Precision:** Exact and lemma matches are resolved deterministically and bypass taxonomies.
2. **Linguistic Soundness:** POS guarding ensures that cross-category word pairs are never evaluated as false synonyms.
3. **WordNet Path Similarity:** Nouns and verbs are evaluated via hierarchical taxonomic paths in a reproducible, calibrated manner.
4. **Transparent Domain Limitations:** Faithfully documents that general WordNet yields low similarity (0.125) for CS domain synonyms like `record` and `row`.

### Strict Pipeline Boundaries
- Stage 6 evaluates **individual terms/tokens only**.
- It does **not** perform triple alignment or graph matching (Stage 7).
- It does **not** perform textual entailment or relevance filtering (Stage 8).
- It does **not** calculate student grades or similarity scores between answers (Stage 9).